# Position-aware similar-player search

Same FBref table as `fbref_test.ipynb` (`player_stats.csv`), but position is a **comparison pool**, not a hard filter.

Change **QUERY** and re-run from that cell down. Scaler / PCA / NMF are fit on the target's pool only, and cached so Bruno → Pedri (both `MF`) does not refit. Keepers are excluded: this extract has no keeper stats.

SID and the autoencoder from `fbref_test.ipynb` are left out here (slow, and they do not change the position logic).

In [1]:
from IPython.display import display
import polars as pl

from player_similarity import (
    POS_COL,
    search,
    summarize,
    top_table,
    consensus_table,
    role_recipe,
    nmf_mix_table,
    nmf_loadings,
)

## Query

Set the target and re-run this cell plus everything below. `team` / `league` disambiguate duplicate names. `allow_adjacent_pos` adds hybrid neighbours (`DF,MF` can search midfielders too). `max_age` limits the comparison pool to players at or below that age (years); the target is always kept for scoring even if older.

In [2]:
QUERY = {
    "player": "Bruno Fernandes",
    "team": None,                 # e.g. "Manchester Utd"
    "league": None,               # e.g. "ENG-Premier League"
    "min_90s": 8.0,
    "max_age": None,              # e.g. 23 — only compare to players aged 23 or younger
    "allow_adjacent_pos": False,
    "top_n": 9,
    "pca_var": 0.80,
    "nmf_k": 5,
    "rrf_k": 60,
}

result = search(QUERY)
summarize(result)
if result["matches"].height > 1:
    display(result["matches"].select("player", "team", "league", POS_COL, "primary_pos"))

target: Bruno Fernandes (Manchester Utd, ENG-Premier League)  pos=MF  pool=MF  n=482  pca=12 (80% var)


In [3]:
print("Role-block scores (mean of RobustScaler features in each block, within the pool)")
display(role_recipe(result))
print("columns used per role:")
for role, cols in result["role_cols"].items():
    shown = ", ".join(cols[:6])
    extra = " …" if len(cols) > 6 else ""
    print(f"  {role}: {len(cols)} — {shown}{extra}")

Role-block scores (mean of RobustScaler features in each block, within the pool)


role,score
str,f64
"""creator""",1.484945
"""carrier""",0.321387
"""ball_winner""",-0.003781
"""box""",0.861887


columns used per role:
  creator: 12 — ('Expected', 'xAG'), ('SCA Types', 'PassLive'), ('SCA Types', 'PassDead'), ('SCA Types', 'TO'), ('SCA Types', 'Sh'), ('SCA Types', 'Fld') …
  carrier: 10 — ('Take-Ons', 'Att'), ('Take-Ons', 'Succ'), ('Take-Ons', 'Tkld'), ('Carries', 'PrgDist'), ('Carries', 'PrgC'), ('Carries', '1/3') …
  ball_winner: 9 — ('Tackles', 'TklW'), ('Tackles', 'Mid 3rd'), ('Challenges', 'Tkl'), ('Challenges', 'Att'), ('Blocks', 'Sh'), ('Blocks', 'Pass') …
  box: 4 — ('Performance', 'Gls'), ('Expected', 'npxG'), ('Touches', 'Att Pen'), ('Carries', 'CPA')


In [4]:
top_n = QUERY["top_n"]
for name, scores in result["scores"].items():
    print(name)
    display(top_table(result["pool"], scores, f"{name.lower()}_score", top_n))

side = {"rank": list(range(1, top_n + 1))}
for name, scores in result["scores"].items():
    order = scores.argsort()[::-1][:top_n]
    side[name] = [result["pool"]["player"][int(i)] for i in order]
print("Top similar players by model")
display(pl.DataFrame(side))

Cosine


player,team,league,"('pos', '')",primary_pos,cosine_score
str,str,str,str,str,f64
"""Imran Louza""","""Watford""","""ENG-EFL Championship""","""MF""","""MF""",0.838235
"""Joey Veerman""","""PSV Eindhoven""","""NED-Eredivisie""","""MF""","""MF""",0.837103
"""Arda Güler""","""Real Madrid""","""ESP-La Liga""","""MF,FW""","""MF""",0.82247
"""Pablo Fornals""","""Betis""","""ESP-La Liga""","""MF""","""MF""",0.761169
"""Nadiem Amiri""","""Mainz 05""","""GER-Bundesliga""","""MF""","""MF""",0.751438
"""Barry Bannan""","""Sheffield Weds""","""ENG-EFL Championship""","""MF""","""MF""",0.743881
"""Nicolò Barella""","""Inter""","""ITA-Serie A""","""MF""","""MF""",0.741758
"""Hakan Çalhanoğlu""","""Inter""","""ITA-Serie A""","""MF""","""MF""",0.741505
"""Angelo Stiller""","""Stuttgart""","""GER-Bundesliga""","""MF""","""MF""",0.723992


KNN_PCA


player,team,league,"('pos', '')",primary_pos,knn_pca_score
str,str,str,str,str,f64
"""Imran Louza""","""Watford""","""ENG-EFL Championship""","""MF""","""MF""",0.225588
"""Joey Veerman""","""PSV Eindhoven""","""NED-Eredivisie""","""MF""","""MF""",0.205808
"""Nadiem Amiri""","""Mainz 05""","""GER-Bundesliga""","""MF""","""MF""",0.194232
"""Arda Güler""","""Real Madrid""","""ESP-La Liga""","""MF,FW""","""MF""",0.192318
"""Pablo Fornals""","""Betis""","""ESP-La Liga""","""MF""","""MF""",0.178979
"""Phil Foden""","""Manchester City""","""ENG-Premier League""","""MF""","""MF""",0.167099
"""Barry Bannan""","""Sheffield Weds""","""ENG-EFL Championship""","""MF""","""MF""",0.1665
"""Enzo Fernández""","""Chelsea""","""ENG-Premier League""","""MF""","""MF""",0.163193
"""Gustavo Hamer""","""Sheffield Utd""","""ENG-EFL Championship""","""MF,FW""","""MF""",0.162387


Spearman


...

player,team,league,"('pos', '')",primary_pos,roleblocks_score
str,str,str,str,str,f64
"""Fares Chaïbi""","""Eint Frankfurt""","""GER-Bundesliga""","""MF""","""MF""",0.989037
"""Anton Stach""","""Leeds United""","""ENG-Premier League""","""MF""","""MF""",0.983408
"""Arda Güler""","""Real Madrid""","""ESP-La Liga""","""MF,FW""","""MF""",0.97754
"""Danel Sinani""","""St. Pauli""","""GER-Bundesliga""","""MF,FW""","""MF""",0.972336
"""Imran Louza""","""Watford""","""ENG-EFL Championship""","""MF""","""MF""",0.970935
"""Hicham Boudaoui""","""Nice""","""FRA-Ligue 1""","""MF""","""MF""",0.97048
"""Marcelino Núñez""","""Ipswich Town""","""ENG-EFL Championship""","""MF""","""MF""",0.96671
"""Yeremi Pino""","""Crystal Palace""","""ENG-Premier League""","""MF""","""MF""",0.966447
"""Pablo Fornals""","""Betis""","""ESP-La Liga""","""MF""","""MF""",0.966046


NMF


player,team,league,"('pos', '')",primary_pos,nmf_score
str,str,str,str,str,f64
"""Gustavo Hamer""","""Sheffield Utd""","""ENG-EFL Championship""","""MF,FW""","""MF""",0.988547
"""Luís Esteves""","""Gil Vicente FC""","""Por-Primeira Liga""","""MF""","""MF""",0.986954
"""Imran Louza""","""Watford""","""ENG-EFL Championship""","""MF""","""MF""",0.979327
"""Arda Güler""","""Real Madrid""","""ESP-La Liga""","""MF,FW""","""MF""",0.971947
"""Medon Berisha""","""Lecce""","""ITA-Serie A""","""MF""","""MF""",0.967729
"""Pablo Fornals""","""Betis""","""ESP-La Liga""","""MF""","""MF""",0.956331
"""Fares Chaïbi""","""Eint Frankfurt""","""GER-Bundesliga""","""MF""","""MF""",0.951066
"""Marcelino Núñez""","""Ipswich Town""","""ENG-EFL Championship""","""MF""","""MF""",0.949354
"""Danel Sinani""","""St. Pauli""","""GER-Bundesliga""","""MF,FW""","""MF""",0.94823


Top similar players by model


rank,Cosine,KNN_PCA,Spearman,RoleBlocks,NMF
i64,str,str,str,str,str
1,"""Imran Louza""","""Imran Louza""","""Pablo Fornals""","""Fares Chaïbi""","""Gustavo Hamer"""
2,"""Joey Veerman""","""Joey Veerman""","""Arda Güler""","""Anton Stach""","""Luís Esteves"""
3,"""Arda Güler""","""Nadiem Amiri""","""Imran Louza""","""Arda Güler""","""Imran Louza"""
4,"""Pablo Fornals""","""Arda Güler""","""Joey Veerman""","""Danel Sinani""","""Arda Güler"""
5,"""Nadiem Amiri""","""Pablo Fornals""","""Barry Bannan""","""Imran Louza""","""Medon Berisha"""
6,"""Barry Bannan""","""Phil Foden""","""Declan Rice""","""Hicham Boudaoui""","""Pablo Fornals"""
7,"""Nicolò Barella""","""Barry Bannan""","""Nicolò Barella""","""Marcelino Núñez""","""Fares Chaïbi"""
8,"""Hakan Çalhanoğlu""","""Enzo Fernández""","""Gustavo Hamer""","""Yeremi Pino""","""Marcelino Núñez"""
9,"""Angelo Stiller""","""Gustavo Hamer""","""Luís Esteves""","""Pablo Fornals""","""Danel Sinani"""


In [5]:
print("RRF consensus")
display(consensus_table(result))

print("NMF recipe (rows sum to 1). Name the types from the loadings below.")
display(nmf_mix_table(result))
print("NMF type loadings (top FBref columns)")
for line in nmf_loadings(result):
    print(line)

RRF consensus


player,team,league,"('pos', '')",primary_pos,rrf_score,Cosine_rank,KNN_PCA_rank,Spearman_rank,RoleBlocks_rank,NMF_rank
str,str,str,str,str,f64,i64,i64,i64,i64,i64
"""Imran Louza""","""Watford""","""ENG-EFL Championship""","""MF""","""MF""",0.079918,1,1,3,5,3
"""Arda Güler""","""Real Madrid""","""ESP-La Liga""","""MF,FW""","""MF""",0.079125,3,4,2,3,4
"""Pablo Fornals""","""Betis""","""ESP-La Liga""","""MF""","""MF""",0.077047,4,5,1,9,6
"""Gustavo Hamer""","""Sheffield Utd""","""ENG-EFL Championship""","""MF,FW""","""MF""",0.070553,15,9,8,26,1
"""Hakan Çalhanoğlu""","""Inter""","""ITA-Serie A""","""MF""","""MF""",0.069521,8,13,11,15,13
"""Joey Veerman""","""PSV Eindhoven""","""NED-Eredivisie""","""MF""","""MF""",0.069512,2,2,4,22,46
"""Marcelino Núñez""","""Ipswich Town""","""ENG-EFL Championship""","""MF""","""MF""",0.068351,16,14,23,7,8
"""Nadiem Amiri""","""Mainz 05""","""GER-Bundesliga""","""MF""","""MF""",0.067463,5,3,35,21,15
"""Giovani Lo Celso""","""Betis""","""ESP-La Liga""","""MF""","""MF""",0.06668,10,11,13,35,11


NMF recipe (rows sum to 1). Name the types from the loadings below.


player,type_1,type_2,type_3,type_4,type_5
str,f64,f64,f64,f64,f64
"""Bruno Fernandes""",0.067176,0.350854,0.075697,0.10764,0.398633
"""Imran Louza""",0.134112,0.299331,0.083122,0.154798,0.328637
"""Arda Güler""",0.072553,0.274744,0.021936,0.190418,0.440349
"""Pablo Fornals""",0.104322,0.242494,0.082575,0.21567,0.354939
"""Gustavo Hamer""",0.061397,0.334781,0.084911,0.173041,0.34587
"""Hakan Çalhanoğlu""",0.198574,0.35363,0.124378,0.0,0.323418
"""Joey Veerman""",0.082176,0.469597,0.211446,0.023351,0.21343
"""Marcelino Núñez""",0.042532,0.218199,0.068993,0.157353,0.512924
"""Nadiem Amiri""",0.069457,0.277721,0.149824,0.229568,0.27343


NMF type loadings (top FBref columns)
type_1: ('Short', 'Cmp%')=16.03; ('Total', 'Cmp%')=14.29; ('Medium', 'Cmp%')=12.86; ('Short', 'Cmp')=11.41; ('Receiving', 'Rec')=11.13; ('Short', 'Att')=11.12; ('Carries', 'Carries')=11.03; ('Medium', 'Cmp')=10.90
type_2: ('SCA Types', 'PassDead')=5.97; ('Long', 'Att')=5.62; ('KP', '')=4.71; ('Long', 'Cmp')=4.40; ('Expected', 'xA')=4.33; ('PPA', '')=3.96; ('Expected', 'xAG')=3.96; ('Total', 'PrgDist')=3.87
type_3: ('Short', 'Cmp%')=7.41; ('Medium', 'Cmp%')=5.74; ('Total', 'Cmp%')=5.68; ('Challenges', 'Tkl%')=5.15; ('Tackles', 'TklW')=4.98; ('Challenges', 'Att')=4.86; ('Challenges', 'Tkl')=4.78; ('Take-Ons', 'Tkld%')=4.58
type_4: ('Carries', 'CPA')=4.20; ('Take-Ons', 'Succ')=3.07; ('Carries', 'PrgC')=3.03; ('Take-Ons', 'Att')=2.82; ('Carries', '1/3')=2.50; ('Take-Ons', 'Tkld')=2.47; ('SCA Types', 'TO')=2.38; ('Carries', 'Dis')=2.01
type_5: ('Short', 'Cmp%')=3.97; ('Total', 'Cmp%')=3.21; ('Medium', 'Cmp%')=3.01; ('Expected', 'npxG')=2.62; ('Expected'

To search someone else, change `QUERY` and re-run from that cell. Examples:

```python
QUERY["player"] = "Virgil van Dijk"
QUERY["team"] = "Liverpool"
```

```python
QUERY["player"] = "Erling Haaland"
QUERY["allow_adjacent_pos"] = False
```

A new position pool refits scaler/PCA/NMF; the same pool reuses the cache.